# Logistic Regression — Base vs Google Trends

**Part 1** trains and evaluates a logistic regression on price + engineered features.  
**Part 2** adds 5 Google Trends features and evaluates independently.  
**Part 3** compares both models head-to-head.

In [22]:
import pathlib
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    f1_score, log_loss, precision_recall_curve, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [23]:
ROOT            = pathlib.Path("../..")
DATA_DIR        = ROOT / "data"
ARTIFACTS_DIR   = pathlib.Path("artifacts")
PREDICTIONS_DIR = pathlib.Path("predictions")

NUMERIC_FEATURES = [
    "price_at_snapshot",
    "price_deviation_from_half",
    "days_before_close",
    "pct_lifetime_elapsed",
    "duration_days",
    "log_volume",
    "price_mean_7d",   "price_volatility_7d",  "price_min_7d",  "price_max_7d",
    "price_change_7d", "price_range_7d",        "price_trend_7d",
    "price_mean_14d",  "price_volatility_14d", "price_min_14d", "price_max_14d",
    "price_change_14d","price_range_14d",       "price_trend_14d",
]
TRENDS_FEATURES      = ["trend_value", "trend_ma4", "trend_change_4w", "trend_spike", "has_trend_data"]
CATEGORICAL_FEATURES = ["category"]
TARGET               = "outcome"

FEATURES_BASE   = NUMERIC_FEATURES + CATEGORICAL_FEATURES
FEATURES_TRENDS = NUMERIC_FEATURES + TRENDS_FEATURES + CATEGORICAL_FEATURES

---
## Load Data

The trends-enriched dataset contains all base features plus the 5 trend columns, split across two parquet files.

In [24]:
df = pd.read_parquet(DATA_DIR / "polymarket_ml_dataset_with_trends_clean.parquet")

df["category"] = df["category"].fillna("other")
df = df.dropna(subset=[TARGET])

train = df[df["split"] == "train"]
test  = df[df["split"] == "test"]

assert len(set(train["market_id"]) & set(test["market_id"])) == 0, "Market leakage detected"

counts         = train.groupby("market_id").size()
sample_weights = train["market_id"].map(counts).rdiv(1).values

y_train    = train[TARGET]
y_test     = test[TARGET]
test_reset = test.reset_index(drop=True)

print(f"Total rows : {len(df):,}  |  Columns: {df.shape[1]}")
print(f"Train      : {len(train):,}  |  {train['market_id'].nunique():,} markets")
print(f"Test       : {len(test):,}   |  {test['market_id'].nunique():,} markets")
print(f"Trend coverage (has_trend_data=1): {df['has_trend_data'].mean():.1%}")
df.head()

Total rows : 1,448,142  |  Columns: 32
Train      : 1,159,652  |  16,774 markets
Test       : 288,490   |  4,174 markets
Trend coverage (has_trend_data=1): 99.6%


,market_id,snapshot_timestamp,days_before_close,pct_lifetime_elapsed,duration_days,price_at_snapshot,price_deviation_from_half,total_volume,log_volume,outcome,...,price_range_14d,price_trend_14d,split,category,question,trend_value,trend_ma4,trend_change_4w,trend_spike,has_trend_data
0,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-01 18:45:42.437000+00:00,59.22,0.1912,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000526,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
1,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-02 06:45:42.437000+00:00,58.72,0.1980,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000354,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
2,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-02 18:45:42.437000+00:00,58.22,0.2049,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000215,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
3,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-03 06:45:42.437000+00:00,57.72,0.2117,73,0.03,0.47,40175.18,10.601,0,...,0.03,0.000479,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
4,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-03 18:45:42.437000+00:00,57.22,0.2185,73,0.03,0.47,40175.18,10.601,0,...,0.03,0.000499,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1


---
## Shared Helpers

In [25]:
def build_pipeline(num_cols, cat_cols):
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ("scaler",  StandardScaler()),
            ]), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ])),
        ("clf", LogisticRegression(
            class_weight="balanced", max_iter=1000,
            solver="lbfgs", C=1.0, random_state=42,
        )),
    ])

def get_threshold(y_true, y_prob):
    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    return float(thresh[np.argmax(f1)]), float(np.max(f1))

def evaluate(y_true, y_prob, label, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "AUC-ROC" : roc_auc_score(y_true, y_prob),
        "PR-AUC"  : average_precision_score(y_true, y_prob),
        "Log-loss": log_loss(y_true, y_prob),
        "Brier"   : brier_score_loss(y_true, y_prob),
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1"      : f1_score(y_true, y_pred),
    }
    print(f"\n{'─'*50}")
    print(f"  {label}  (threshold={threshold:.3f})")
    print(f"{'─'*50}")
    for name, val in metrics.items():
        print(f"  {name:<10}: {val:.4f}")
    print(f"{'─'*50}")
    return metrics

def per_category(test_df, y_prob, threshold):
    rows = []
    for cat in sorted(test_df["category"].unique()):
        mask = test_df["category"] == cat
        yt   = test_df.loc[mask, TARGET].values
        if len(np.unique(yt)) < 2 or len(yt) < 10:
            continue
        yp = y_prob[mask.values]
        rows.append({
            "category": cat,
            "n"       : int(mask.sum()),
            "AUC"     : roc_auc_score(yt, yp),
            "PR-AUC"  : average_precision_score(yt, yp),
            "F1"      : f1_score(yt, (yp >= threshold).astype(int)),
            "YES%"    : float(yt.mean()),
        })
    return pd.DataFrame(rows).set_index("category").sort_values("AUC", ascending=False)

def market_eval(test_df, y_prob, threshold):
    mdf = (
        test_df.assign(pred_prob=y_prob)
        .groupby("market_id")
        .agg(pred_prob=("pred_prob", "mean"), outcome=(TARGET, "first"))
        .reset_index()
    )
    mp, mt = mdf["pred_prob"].values, mdf["outcome"].values
    mpred  = (mp >= threshold).astype(int)
    print(f"Market-level evaluation ({len(mdf):,} markets)")
    print(f"  AUC-ROC  : {roc_auc_score(mt, mp):.4f}")
    print(f"  PR-AUC   : {average_precision_score(mt, mp):.4f}")
    print(f"  Brier    : {brier_score_loss(mt, mp):.4f}")
    print(f"  Accuracy : {accuracy_score(mt, mpred):.4f}")
    print(f"  F1       : {f1_score(mt, mpred):.4f}")
    return {"AUC-ROC": roc_auc_score(mt,mp), "PR-AUC": average_precision_score(mt,mp),
            "Brier": brier_score_loss(mt,mp), "Accuracy": accuracy_score(mt,mpred), "F1": f1_score(mt,mpred)}

---
## Baseline — Market Price

The simplest predictor: use `price_at_snapshot` directly as the probability.
This is the crowd's consensus — a useful sanity check for any model we build.

In [26]:
y_prob_baseline        = test_reset["price_at_snapshot"].values
thresh_bl, f1_bl       = get_threshold(y_test, y_prob_baseline)
print(f"Baseline threshold: {thresh_bl:.3f}  |  F1: {f1_bl:.4f}")

metrics_baseline_row = evaluate(y_test, y_prob_baseline, "Baseline (market price)", thresh_bl)
metrics_baseline_mkt = market_eval(test_reset, y_prob_baseline, thresh_bl)
cat_baseline         = per_category(test_reset, y_prob_baseline, thresh_bl)

Baseline threshold: 0.500  |  F1: 0.6699

──────────────────────────────────────────────────
  Baseline (market price)  (threshold=0.500)
──────────────────────────────────────────────────
  AUC-ROC   : 0.8890
  PR-AUC    : 0.7440
  Log-loss  : 0.3278
  Brier     : 0.1004
  Accuracy  : 0.8703
  F1        : 0.6699
──────────────────────────────────────────────────
Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9420
  PR-AUC   : 0.8283
  Brier    : 0.0652
  Accuracy : 0.9178
  F1       : 0.7351


---
# Part 1 — Base Model

Price + engineered features only, no Google Trends.

## 1.1 Train

In [27]:
pipe_base = build_pipeline(
    num_cols=NUMERIC_FEATURES,
    cat_cols=CATEGORICAL_FEATURES,
)
pipe_base.fit(train[FEATURES_BASE], y_train, clf__sample_weight=sample_weights)

y_prob_base        = pipe_base.predict_proba(test[FEATURES_BASE])[:, 1]
thresh_base, f1_base = get_threshold(y_test, y_prob_base)
print(f"Optimal threshold: {thresh_base:.3f}  |  F1: {f1_base:.4f}")

Optimal threshold: 0.760  |  F1: 0.6653


## 1.2 Row-Level Evaluation

In [28]:
metrics_base_row = evaluate(y_test, y_prob_base, "Base Model", thresh_base)


──────────────────────────────────────────────────
  Base Model  (threshold=0.760)
──────────────────────────────────────────────────
  AUC-ROC   : 0.8881
  PR-AUC    : 0.7210
  Log-loss  : 0.4644
  Brier     : 0.1423
  Accuracy  : 0.8592
  F1        : 0.6653
──────────────────────────────────────────────────


## 1.3 Per-Category Breakdown

In [29]:
cat_base = per_category(test_reset, y_prob_base, thresh_base)
cat_base.style.format({"AUC": "{:.4f}", "PR-AUC": "{:.4f}", "F1": "{:.4f}", "YES%": "{:.1%}"})

,n,AUC,PR-AUC,F1,YES%
category,,,,,
geopolitics,16578,0.9356,0.7406,0.6664,13.8%
entertainment,37027,0.9190,0.7265,0.6497,14.9%
finance,25093,0.9169,0.8064,0.7083,24.3%
politics_global,22591,0.9073,0.7377,0.6776,22.3%
politics_us,52699,0.9052,0.7649,0.7308,24.7%
crypto,24586,0.9036,0.7266,0.7161,26.1%
science_tech,15530,0.8855,0.7142,0.6556,17.8%
other,1364,0.8684,0.3088,0.4386,4.7%
sports,93022,0.8428,0.6611,0.5831,20.9%


## 1.4 Market-Level Evaluation

In [30]:
metrics_base_mkt = market_eval(test_reset, y_prob_base, thresh_base)

Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9380
  PR-AUC   : 0.8070
  Brier    : 0.0900
  Accuracy : 0.9116
  F1       : 0.7228


## 1.5 Feature Importance

In [31]:
clf_step  = pipe_base.named_steps["clf"]
prep_step = pipe_base.named_steps["preprocessor"]
cat_names = list(prep_step.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
all_names = NUMERIC_FEATURES + cat_names

importance_df_base = (
    pd.DataFrame({"feature": all_names, "coefficient": clf_step.coef_[0]})
    .assign(abs_coef=lambda d: d["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
    .reset_index(drop=True)
)

importance_df_base.head(20).style.bar(
    subset=["coefficient"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,feature,coefficient
0,price_at_snapshot,0.969753
1,log_volume,0.490603
2,duration_days,-0.469008
3,price_max_14d,0.441962
4,days_before_close,0.417625
5,price_min_14d,0.390303
6,category_other,0.383470
7,category_geopolitics,-0.315402
8,category_sports,-0.228272
9,price_range_14d,0.204321


---
# Part 2 — Model with Google Trends

Same as Part 1 plus 5 Google Trends features: `trend_value`, `trend_ma4`, `trend_change_4w`, `trend_spike`, `has_trend_data`.

## 2.1 Train

In [32]:
pipe_trends = build_pipeline(
    num_cols=NUMERIC_FEATURES + TRENDS_FEATURES,
    cat_cols=CATEGORICAL_FEATURES,
)
pipe_trends.fit(train[FEATURES_TRENDS], y_train, clf__sample_weight=sample_weights)

y_prob_trends          = pipe_trends.predict_proba(test[FEATURES_TRENDS])[:, 1]
thresh_trends, f1_trends = get_threshold(y_test, y_prob_trends)
print(f"Optimal threshold: {thresh_trends:.3f}  |  F1: {f1_trends:.4f}")

Optimal threshold: 0.781  |  F1: 0.6651


## 2.2 Row-Level Evaluation

In [33]:
metrics_trends_row = evaluate(y_test, y_prob_trends, "Trends Model", thresh_trends)


──────────────────────────────────────────────────
  Trends Model  (threshold=0.781)
──────────────────────────────────────────────────
  AUC-ROC   : 0.8881
  PR-AUC    : 0.7203
  Log-loss  : 0.4640
  Brier     : 0.1422
  Accuracy  : 0.8619
  F1        : 0.6651
──────────────────────────────────────────────────


## 2.3 Per-Category Breakdown

In [34]:
cat_trends = per_category(test_reset, y_prob_trends, thresh_trends)
cat_trends.style.format({"AUC": "{:.4f}", "PR-AUC": "{:.4f}", "F1": "{:.4f}", "YES%": "{:.1%}"})

,n,AUC,PR-AUC,F1,YES%
category,,,,,
geopolitics,16578,0.9354,0.7385,0.6596,13.8%
entertainment,37027,0.9190,0.7270,0.6484,14.9%
finance,25093,0.9168,0.8065,0.7070,24.3%
crypto,24586,0.9062,0.7317,0.7151,26.1%
politics_global,22591,0.9061,0.7300,0.6802,22.3%
politics_us,52699,0.9048,0.7653,0.7379,24.7%
science_tech,15530,0.8857,0.7113,0.6521,17.8%
other,1364,0.8677,0.3079,0.4587,4.7%
sports,93022,0.8426,0.6609,0.5785,20.9%


## 2.4 Market-Level Evaluation

In [35]:
metrics_trends_mkt = market_eval(test_reset, y_prob_trends, thresh_trends)

Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9379
  PR-AUC   : 0.8067
  Brier    : 0.0900
  Accuracy : 0.9128
  F1       : 0.7221


## 2.5 Feature Importance

Trend features are highlighted in yellow — their magnitude relative to price features shows how much signal they add.

In [36]:
clf_step  = pipe_trends.named_steps["clf"]
prep_step = pipe_trends.named_steps["preprocessor"]
cat_names = list(prep_step.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
all_names = NUMERIC_FEATURES + TRENDS_FEATURES + cat_names

importance_df = (
    pd.DataFrame({"feature": all_names, "coefficient": clf_step.coef_[0]})
    .assign(
        abs_coef = lambda d: d["coefficient"].abs(),
        is_trend = lambda d: d["feature"].isin(TRENDS_FEATURES),
    )
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
    .reset_index(drop=True)
)

print("Trend feature coefficients:")
print(importance_df[importance_df["is_trend"]].to_string(index=False))
print()

importance_df.head(25).style.bar(
    subset=["coefficient"], align="zero", color=["#d65f5f", "#5fba7d"]
).apply(
    lambda col: ["background-color: #fff3cd" if v else "" for v in importance_df.head(25)["is_trend"]],
    axis=0, subset=["feature", "coefficient"]
)

Trend feature coefficients:
        feature  coefficient  is_trend
    trend_value     0.327737      True
      trend_ma4    -0.240601      True
trend_change_4w    -0.106904      True
 has_trend_data    -0.039430      True
    trend_spike     0.033687      True



,feature,coefficient,is_trend
0,price_at_snapshot,0.941145,False
1,log_volume,0.494312,False
2,duration_days,-0.455420,False
3,price_max_14d,0.438335,False
4,days_before_close,0.403496,False
5,price_min_14d,0.388474,False
6,trend_value,0.327737,True
7,trend_ma4,-0.240601,True
8,category_geopolitics,-0.214738,False
9,category_politics_us,0.211582,False


---
# Part 3 — Comparison

Head-to-head: Base vs Trends across row-level metrics, market-level metrics, and per-category AUC.

## 3.1 Row-Level

In [37]:
row_comparison = pd.DataFrame({
    "Baseline": metrics_baseline_row,
    "Base":     metrics_base_row,
    "Trends":   metrics_trends_row,
})
row_comparison["Δ Base"]   = row_comparison["Base"]   - row_comparison["Baseline"]
row_comparison["Δ Trends"] = row_comparison["Trends"] - row_comparison["Baseline"]
row_comparison.style.format("{:.4f}").bar(
    subset=["Δ Base", "Δ Trends"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,Baseline,Base,Trends,Δ Base,Δ Trends
AUC-ROC,0.8890,0.8881,0.8881,-0.0010,-0.0009
PR-AUC,0.7440,0.7210,0.7203,-0.0230,-0.0237
Log-loss,0.3278,0.4644,0.4640,0.1365,0.1361
Brier,0.1004,0.1423,0.1422,0.0419,0.0418
Accuracy,0.8703,0.8592,0.8619,-0.0111,-0.0085
F1,0.6699,0.6653,0.6651,-0.0046,-0.0047


## 3.2 Market-Level

In [38]:
mkt_comparison = pd.DataFrame({
    "Baseline": metrics_baseline_mkt,
    "Base":     metrics_base_mkt,
    "Trends":   metrics_trends_mkt,
})
mkt_comparison["Δ Base"]   = mkt_comparison["Base"]   - mkt_comparison["Baseline"]
mkt_comparison["Δ Trends"] = mkt_comparison["Trends"] - mkt_comparison["Baseline"]
mkt_comparison.style.format("{:.4f}").bar(
    subset=["Δ Base", "Δ Trends"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,Baseline,Base,Trends,Δ Base,Δ Trends
AUC-ROC,0.9420,0.9380,0.9379,-0.0040,-0.0041
PR-AUC,0.8283,0.8070,0.8067,-0.0214,-0.0216
Brier,0.0652,0.0900,0.0900,0.0249,0.0248
Accuracy,0.9178,0.9116,0.9128,-0.0062,-0.0050
F1,0.7351,0.7228,0.7221,-0.0124,-0.0130


## 3.3 Per-Category AUC Delta

In [39]:
cat_comparison = cat_baseline[["n", "AUC", "YES%"]].rename(columns={"AUC": "AUC (baseline)"})
cat_comparison["AUC (base)"]   = cat_base["AUC"]
cat_comparison["AUC (trends)"] = cat_trends["AUC"]
cat_comparison["Δ Base"]       = cat_comparison["AUC (base)"]   - cat_comparison["AUC (baseline)"]
cat_comparison["Δ Trends"]     = cat_comparison["AUC (trends)"] - cat_comparison["AUC (baseline)"]
(
    cat_comparison
    .sort_values("AUC (baseline)", ascending=False)
    .style
    .format("{:.4f}", subset=["AUC (baseline)", "AUC (base)", "AUC (trends)", "Δ Base", "Δ Trends"])
    .format("{:.1%}", subset=["YES%"])
    .bar(subset=["Δ Base", "Δ Trends"], align="zero", color=["#d65f5f", "#5fba7d"])
)

,n,AUC (baseline),YES%,AUC (base),AUC (trends),Δ Base,Δ Trends
category,,,,,,,
geopolitics,16578,0.9265,13.8%,0.9356,0.9354,0.0091,0.0089
finance,25093,0.9183,24.3%,0.9169,0.9168,-0.0014,-0.0015
entertainment,37027,0.9101,14.9%,0.9190,0.9190,0.0089,0.0089
politics_us,52699,0.9083,24.7%,0.9052,0.9048,-0.0031,-0.0035
politics_global,22591,0.9057,22.3%,0.9073,0.9061,0.0016,0.0004
crypto,24586,0.9052,26.1%,0.9036,0.9062,-0.0016,0.0009
science_tech,15530,0.8876,17.8%,0.8855,0.8857,-0.0022,-0.0020
other,1364,0.8847,4.7%,0.8684,0.8677,-0.0163,-0.0170
sports,93022,0.8400,20.9%,0.8428,0.8426,0.0028,0.0026


---
## Save Artifacts

In [40]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": pipe_base},   ARTIFACTS_DIR / "model_base.joblib")
joblib.dump({"pipeline": pipe_trends}, ARTIFACTS_DIR / "model_trends.joblib")
print(f"Models saved → {ARTIFACTS_DIR}")

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
pred_df = test[["market_id", "category", TARGET]].copy().reset_index(drop=True)
pred_df["pred_prob_base"]    = y_prob_base
pred_df["pred_prob_trends"]  = y_prob_trends
pred_df["pred_label_base"]   = (y_prob_base   >= thresh_base).astype(int)
pred_df["pred_label_trends"] = (y_prob_trends >= thresh_trends).astype(int)
preds_path = PREDICTIONS_DIR / "predictions.csv"
pred_df.to_csv(preds_path, index=False)
print(f"Predictions saved → {preds_path}")
pred_df.head()

Models saved → artifacts
Predictions saved → predictions/predictions.csv


,market_id,category,outcome,pred_prob_base,pred_prob_trends,pred_label_base,pred_label_trends
0,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.763279,0.748789,1,0
1,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.821828,0.809155,1,1
2,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.826740,0.816410,1,1
3,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.828752,0.817975,1,1
4,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.910018,0.903860,1,1
